# 02 · Supervised learning and discharge forecasts

At the end of today, can we forecast tomorrow's daily mean discharge? We'll compare a forecast that repeats today's flow with a fitted straight line. Both models see the same information at forecast time.

The readings are synthetic, in m³/s, with a few peaks and recessions. Use the short **Try it** cells as you go. Expand **Show solution** after each attempt. The worked cells can run in order even if you skip a task.

## 1 · Turn a hydrograph into examples

**Supervised learning** means fitting a rule using examples with known answers. A **feature** is an input available when the forecast is issued. A **target** (also called a label) is the answer we want to predict. Here today's measured flow is the feature and tomorrow's measured flow is the target. Predicting a numerical value is called **regression**. The one-day gap is the **forecast horizon**.

A **lag** is how far back an input is measured relative to the target. For a day 3 target, day 2 flow is lag 1. One **sample** is an input and its matching target. For the first sample, day 1 was 2 and day 2 was 3, so the pair is `(2, 3)`.

**Try it.** Predict the second pair and the number of pairs you can make from four consecutive days.

In [1]:
import torch

q = torch.tensor([2., 3., 7., 5., 3., 2., 4., 9., 6., 4., 3., 2., 4., 8., 8., 7., 6.])
print(q[:4])  # The first four days.
# Write your prediction here.

tensor([2., 3., 7., 5.])


<details>
<summary>Show solution</summary>

The second input is day 2 flow (3), with day 3 flow (7) as its target. Four days give three pairs because the last day has no next-day target in this subset.

</details>

`q[:-1]` drops the final reading and `q[1:]` drops the first. At each index, the two slices now line up one day apart. `unsqueeze(1)` inserts a length-one feature axis so we can think in terms of rows (samples) and columns (features).

In [2]:
x = q[:-1].unsqueeze(1)  # (17,) → (16,) → (16, 1)
y = q[1:]               # (17,) → (16,), one target per sample
target_days = torch.arange(2, len(q) + 1)  # (16,), day numbers for the rows

print("Feature and target shapes", tuple(x.shape), tuple(y.shape))
for i in range(3):
    print(f"Day {target_days[i] - 1} flow {x[i, 0]} → day {target_days[i]} flow {y[i]}")

Feature and target shapes (16, 1) (16,)
Day 1 flow 2.0 → day 2 flow 3.0
Day 2 flow 3.0 → day 3 flow 7.0
Day 3 flow 7.0 → day 4 flow 5.0


## 2 · Separate past and future

The **training set** is where we fit model parameters. The **validation set** helps us compare candidate models or settings. The **test set** is reserved for a final check after those choices. **Generalisation** means doing well on data the model hasn't been fitted to.

**Data leakage** happens when information unavailable at forecast time influences the model or its evaluation. For example, fitting on later dates before evaluating an earlier forecast gives the model access to the future. We'll split by target day. The last training day's observed flow may be the first validation feature because it's available when that forecast is issued. For spatial transfer, we'd ask a different question and hold out entire catchments.

**Try it.** The first ten samples are for training and the next three are for validation. Which day is the first validation target, and which day's flow can you use for its prediction?

In [ ]:
# Write your prediction here. Use target_days to check it.
# val target = day 13
# val prediction = day 12 flow = 4.0

<details>
<summary>Show solution</summary>

The first validation target is day 12. Its feature is day 11 flow, observed before issuing the day 12 forecast.

```python
print(target_days[10], x[10, 0])
```

</details>

In [3]:
train_end = 10
val_end = 13
x_train, y_train = x[:train_end], y[:train_end]  # (10, 1) and (10,)
x_val, y_val = x[train_end:val_end], y[train_end:val_end]  # (3, 1) and (3,)
x_test, y_test = x[val_end:], y[val_end:]  # (3, 1) and (3,)
print("Training days", target_days[:train_end])
print("Validation days", target_days[train_end:val_end])
print("Test days", target_days[val_end:])

Training days tensor([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
Validation days tensor([12, 13, 14])
Test days tensor([15, 16, 17])


## 3 · Score a persistence forecast

A **baseline** is a reference method a more complex model should justify improving on. **Persistence** predicts that tomorrow's flow equals today's. It can work well on slow recessions and can miss a rapid rise.

A **prediction** is the model's estimate. Here we use the signed error `prediction - target`. **Mean absolute error (MAE)** averages the sizes of the misses, so its units are m³/s. For a prediction of 3 and a target of 7, the signed error is −4 and its absolute error is 4.

**Try it.** Use the validation inputs to make persistence predictions. Check the shape before subtracting. A column of shape `(3, 1)` minus a vector of shape `(3,)` broadcasts to `(3, 3)`, giving nine comparisons where we need three.

In [4]:
# TODO: Select the feature as a vector, then subtract y_val, take abs(), and average.
x_val_vector = x_val[:, 0]  # (3, 1) → (3,)
x_val_loss = (x_val_vector - y_val).abs().mean()  # (3,) → () scalar
print("Validation loss", x_val_loss)

Validation loss tensor(2.3333)


<details>
<summary>Show solution</summary>

Selecting the column with an integer gives shape `(3,)`. Subtraction and absolute value keep that shape. The mean reduces it to a scalar (2.33 m³/s).

```python
persistence_val = x_val[:, 0]
mae = (persistence_val - y_val).abs().mean()
print(mae.item())
```

</details>

In [5]:
def mae(predictions, targets):
    return (predictions - targets).abs().mean()

persistence_val = x_val[:, 0]  # (3, 1) → (3,)
persistence_val_mae = mae(persistence_val, y_val)
print(f"Persistence validation MAE: {persistence_val_mae:.2f} m³/s")

Persistence validation MAE: 2.33 m³/s


## 4 · Fit a straight line

A **model** maps features to predictions. A **parameter** is a value learnt from training data. Here we predict `slope × today's flow + intercept`. The slope controls how much the forecast changes when today's flow rises. The intercept is the prediction when today's flow is zero.

To fit the line we minimise the sum of squared training errors. This quantity is the **training objective**, often called a **loss**. A **metric** is a score we report to judge predictions. We use MAE for that here. Squared errors give large misses more weight than absolute errors do. PyTorch's least-squares solver fits the line directly. Lesson 3 will show how gradient descent updates parameters one step at a time.

The solver needs one column per parameter. The discharge column is multiplied by the slope and the column of ones is multiplied by the intercept. `torch.ones_like` makes a same-shaped tensor filled with ones, and `torch.cat(..., dim=1)` joins columns.

In [6]:
design_train = torch.cat((x_train, torch.ones_like(x_train)), dim=1)  # (10, 2)
parameters = torch.linalg.lstsq(design_train, y_train).solution  # (2,)
slope, intercept = parameters
print(f"Slope {slope:.2f}, intercept {intercept:.2f} m³/s")

Slope 0.22, intercept 3.63 m³/s


**Try it.** Write the prediction formula for each validation row. Which model has the lower validation MAE here? Inspect the day with the largest observed validation flow and compare the two signed errors.

In [7]:
# TODO: Make a vector of linear predictions and compare it with persistence_val.
predictions_val = slope * x_val[:, 0] + intercept  # (3,)
linear_val_mae = mae(predictions_val, y_val)
print(f"Linear validation MAE: {linear_val_mae:.2f} m³/s")

val_max_idx = torch.argmax(y_val)  # Index of the maximum value in y_val
val_max_day = target_days[train_end + val_max_idx]  # Day number of
pred_val_se = predictions_val[val_max_idx] - y_val[val_max_idx]  # Prediction error on that day
pers_val_se = persistence_val[val_max_idx] - y_val[val_max_idx]  # Persistence error on that day
print(f"Day {val_max_day} max flow {y_val[val_max_idx]} m³/s, linear prediction error {pred_val_se:.2f} m³/s, persistence error {pers_val_se:.2f} m³/s")

Linear validation MAE: 1.95 m³/s
Day 14 max flow 8.0 m³/s, linear prediction error -3.51 m³/s, persistence error -4.00 m³/s


<details>
<summary>Show solution</summary>

Both forecasts have shape `(3,)`. The line has lower validation MAE on these three days. On the rising flow that peaks at 8 m³/s, both models underpredict.

```python
linear_val = slope * x_val[:, 0] + intercept
print(mae(linear_val, y_val), persistence_val_mae)
peak = y_val.argmax()
print(target_days[train_end:val_end][peak], linear_val[peak] - y_val[peak],
      persistence_val[peak] - y_val[peak])
```

</details>

## 5 · Check the held-out period

The validation scores help us choose between the two models. **Overfitting** means learning patterns specific to the data used for fitting or selection so that performance suffers on new data. Here each evaluation period covers only three targets. Even a clear difference in MAE can depend on which part of the hydrograph lands in each split.

**Try it.** Choose a model by validation MAE. Then score that model and the persistence baseline on the test days without refitting. Does your choice improve on the baseline on these particular days?

In [8]:
# TODO: Choose using the validation scores, then compute the two test scores.
persistence_test = x_test[:, 0]  # (3, 1) → (3,)
persistence_test_mae = mae(persistence_test, y_test)
predictions_test = slope * x_test[:, 0] + intercept  # (3,)
linear_test_mae = mae(predictions_test, y_test)
print(f"Persistence test MAE: {persistence_test_mae:.2f} m³/s, linear test MAE: {linear_test_mae:.2f} m³/s")

Persistence test MAE: 0.67 m³/s, linear test MAE: 1.72 m³/s


<details>
<summary>Show solution</summary>

The line wins on validation. On these test targets persistence has the lower MAE. A short period can give a different ranking if it covers different flow behaviour.

```python
linear_val = slope * x_val[:, 0] + intercept
persistence_test = x_test[:, 0]
if mae(linear_val, y_val) < persistence_val_mae:
    chosen_test = slope * x_test[:, 0] + intercept
else:
    chosen_test = persistence_test
print(f"Chosen model test MAE: {mae(chosen_test, y_test):.2f} m³/s")
print(f"Persistence test MAE: {mae(persistence_test, y_test):.2f} m³/s")
```

</details>

For extra practice, run `python 02_discharge_baselines.py` and edit the synthetic series. Before seeing the new scores, predict what adding a sharp rise in the test period will do to the persistence forecast.